# Part A: Base CNN Model with TensorFlow High-Level API
## Standard TensorFlow Practice - No Manual Training Loops

This notebook demonstrates:
- TensorFlow/Keras high-level APIs
- tf.data API for efficient data pipelines
- Automatic GPU management
- model.fit() training (NO manual loops)
- Built-in callbacks and optimization

## 1. Install and Import Libraries

In [ ]:
# Install required packages
!pip install tensorflow>=2.13 tensorflow-gpu torch torchvision torchinfo -q

import tensorflow as tf
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)

## 2. GPU Configuration and Detection

In [ ]:
# Check TensorFlow GPU availability
print("\n=== TensorFlow GPU Configuration ===")
tf_gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow GPU Available: {len(tf_gpus) > 0}")
if tf_gpus:
    for gpu in tf_gpus:
        print(f"  - {gpu}")

# Configure TensorFlow to use GPU memory growth
for gpu in tf_gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Check PyTorch GPU availability
print("\n=== PyTorch GPU Configuration ===")
print(f"PyTorch GPU Available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA Version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

# Set default device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Default PyTorch Device: {device}")

## 3. Load and Prepare Dataset

In [ ]:
# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten labels
y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"\nDataset shapes:")
print(f"  Training images: {x_train.shape}")
print(f"  Training labels: {y_train.shape}")
print(f"  Test images: {x_test.shape}")
print(f"  Test labels: {y_test.shape}")

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(class_names)
print(f"\nNumber of classes: {num_classes}")
print(f"Class names: {class_names}")

## 4. Create tf.data Pipeline with Augmentation

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
IMG_SIZE = 32

def create_dataset(x, y, batch_size, augment=False):
    """
    Create an optimized tf.data.Dataset pipeline.
    
    Args:
        x: Input images
        y: Labels
        batch_size: Batch size
        augment: Whether to apply data augmentation
    
    Returns:
        Optimized tf.data.Dataset
    """
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    
    if augment:
        # Data augmentation for training
        def augment_fn(image, label):
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
            image = tf.image.random_brightness(image, 0.2)
            image = tf.image.random_contrast(image, 0.8, 1.2)
            image = tf.clip_by_value(image, 0.0, 1.0)
            return image, label
        
        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Batch and prefetch for GPU optimization
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create training and validation datasets
# Split: 90% train, 10% validation
num_train = int(0.9 * len(x_train))
indices = np.random.permutation(len(x_train))

x_train_split = x_train[indices[:num_train]]
y_train_split = y_train[indices[:num_train]]

x_val = x_train[indices[num_train:]]
y_val = y_train[indices[num_train:]]

# Create tf.data datasets
train_dataset = create_dataset(x_train_split, y_train_split, BATCH_SIZE, augment=True)
val_dataset = create_dataset(x_val, y_val, BATCH_SIZE, augment=False)
test_dataset = create_dataset(x_test, y_test, BATCH_SIZE, augment=False)

print(f"Training dataset batches: {len(train_dataset)}")
print(f"Validation dataset batches: {len(val_dataset)}")
print(f"Test dataset batches: {len(test_dataset)}")

## 5. Build Baseline CNN Model

## 6. Compile Model with Optimizers and Loss Functions

In [ ]:
# Compile model using high-level API
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

print("Model compiled successfully!")
print(f"\nOptimizer: Adam (learning_rate=0.001)")
print(f"Loss: SparseCategoricalCrossentropy")
print(f"Metrics: Accuracy, Precision, Recall")

## 7. Define Callbacks for Training

In [ ]:
# Create callbacks for better training management
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_baseline_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configured:")
print("  - EarlyStopping (patience=10)")
print("  - ReduceLROnPlateau (factor=0.5, patience=5)")
print("  - ModelCheckpoint (save best model)")

## 8. Train Model Using High-Level API (NO Manual Loops)

In [ ]:
# Train model using model.fit() - NO manual training loops
print("\n=== Starting Training ===")
print(f"Device: {device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training samples: {len(x_train_split)}")
print(f"Validation samples: {len(x_val)}")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

print("\n=== Training Completed ===")

## 9. Evaluate Model on Test Set

In [ ]:
# Evaluate on test set
print("\n=== Model Evaluation ===")
test_results = model.evaluate(test_dataset, verbose=1)

print(f"\nTest Loss: {test_results[0]:.4f}")
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Test Precision: {test_results[2]:.4f}")
print(f"Test Recall: {test_results[3]:.4f}")

## 10. Generate Predictions and Classification Report

In [ ]:
# Generate predictions
y_pred_probs = model.predict(test_dataset, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy: {accuracy:.4f}")

# Classification report
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

## 11. Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history plot saved!")

## 12. Visualize Confusion Matrix

## 13. Visualize Sample Predictions